In [ ]:
# Install all libraries required for our machine learning project.


!pip install  pandas numpy matplotlib seaborn scikit-learn xgboost joblib streamlit

In [ ]:
# Import pandas for loading and manipulating the dataset.
import pandas as pd

# Import NumPy for numerical calculations.
import numpy as np

# Import Matplotlib for creating graphs.
import matplotlib.pyplot as plt

# Import Seaborn for statistical visualizations.
import seaborn as sns

# Import Scikit-learn for machine learning and preprocessing.
import sklearn

# Import XGBoost for our later model comparison.
import xgboost

# Import Joblib for saving our trained model.
import joblib

# Import Streamlit for the final prediction application.
import streamlit as st

# Display a confirmation message if all imports work successfully.
print("Environment setup completed successfully!")

# Display important library versions for reproducibility.
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)

In [ ]:
# Load the CSV file from the same folder as this notebook.
df = pd.read_csv("online_gaming_behavior_dataset.csv")

# Display the first 5 records.
df.head()

In [ ]:
# Show the number of rows and columns.
# Expected shape is approximately (40034, 13).
print("Dataset Shape:", df.shape)

# Display all column names.
print("\nColumn Names:")
print(df.columns.tolist())

In [ ]:
# Display the data type of every column and the number of non-null values.
# This helps us identify numerical and categorical columns.
df.info()

In [ ]:
# Count missing values in every column.
# A value greater than 0 means that column contains missing data.
print("Missing Values:")
print(df.isnull().sum())

In [ ]:
# Count completely duplicated rows in the dataset.
# Duplicate records can affect model training if they are not handled properly.
print("Duplicate Rows:", df.duplicated().sum())

In [ ]:
# Count how many players belong to each EngagementLevel.
# This tells us whether Low, Medium, and High classes are balanced.
print("Engagement Level Distribution:")
print(df["EngagementLevel"].value_counts())

# Show the percentage of each engagement class.
print("\nEngagement Level Percentage:")
print(df["EngagementLevel"].value_counts(normalize=True) * 100)

In [ ]:
# Create a figure with a suitable size.
plt.figure(figsize=(7, 5))

# Count players in each engagement category.
sns.countplot(data=df, x="EngagementLevel")

# Add a title explaining the chart.
plt.title("Player Engagement Level Distribution")

# Label the x-axis.
plt.xlabel("Engagement Level")

# Label the y-axis.
plt.ylabel("Number of Players")

# Display the chart.
plt.show()

In [ ]:
# Create a boxplot comparing play time across engagement levels.
plt.figure(figsize=(8, 5))

# Draw one box for each engagement category.
sns.boxplot(data=df, x="EngagementLevel", y="PlayTimeHours")

# Add a descriptive title.
plt.title("Play Time Hours by Engagement Level")

# Label the x-axis.
plt.xlabel("Engagement Level")

# Label the y-axis.
plt.ylabel("Play Time (Hours)")

# Display the chart.
plt.show()

In [ ]:
# Create a boxplot showing weekly sessions for each engagement category.
plt.figure(figsize=(8, 5))

# Compare the distribution of sessions per week between engagement levels.
sns.boxplot(data=df, x="EngagementLevel", y="SessionsPerWeek")

# Add a descriptive title.
plt.title("Sessions Per Week by Engagement Level")

# Label the x-axis.
plt.xlabel("Engagement Level")

# Label the y-axis.
plt.ylabel("Sessions Per Week")

# Display the chart.
plt.show()

In [ ]:
# Create a boxplot comparing average session duration.
plt.figure(figsize=(8, 5))

# Compare session duration across the three engagement levels.
sns.boxplot(
    data=df,
    x="EngagementLevel",
    y="AvgSessionDurationMinutes"
)

# Add a descriptive title.
plt.title("Average Session Duration by Engagement Level")

# Label the x-axis.
plt.xlabel("Engagement Level")

# Label the y-axis.
plt.ylabel("Average Session Duration (Minutes)")

# Display the chart.
plt.show()

In [ ]:
# Select only numerical columns because correlation requires numerical values.
numeric_df = df.select_dtypes(include=["int64", "float64"])

# Calculate the correlation between numerical variables.
correlation_matrix = numeric_df.corr()

# Create a larger figure so the heatmap is easy to read.
plt.figure(figsize=(10, 7))

# Display the correlation matrix as a heatmap.
# annot=True writes the correlation values inside the cells.
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

# Add a descriptive title.
plt.title("Correlation Between Numerical Features")

# Display the chart.
plt.show()

In [ ]:
# Display the unique values in GameDifficulty.
# We need to confirm the actual category names before creating the ordinal mapping.
print("GameDifficulty values:")
print(df["GameDifficulty"].unique())

# Display the unique values in the target column.
# This confirms the three engagement categories.
print("\nEngagementLevel values:")
print(df["EngagementLevel"].unique())

In [ ]:
# Import ColumnTransformer so different preprocessing can be applied
# to different groups of columns.
from sklearn.compose import ColumnTransformer

# Import Pipeline so preprocessing and the ML model can later be
# connected into one complete workflow.
from sklearn.pipeline import Pipeline

# Import OneHotEncoder for nominal categorical variables.
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Import OrdinalEncoder for categories that have a meaningful order.
from sklearn.preprocessing import OrdinalEncoder


# Create the input features by removing the target column.
X = df.drop(columns=["EngagementLevel"])

# Remove PlayerID because it is only an identifier.
# A player's ID does not represent a meaningful behavioral characteristic.
X = X.drop(columns=["PlayerID"])

# Create the target variable separately.
y = df["EngagementLevel"]


# Define the numerical features.
# These features contain actual numerical measurements/counts.
numeric_features = [
    "Age",
    "PlayTimeHours",
    "InGamePurchases",
    "SessionsPerWeek",
    "AvgSessionDurationMinutes",
    "PlayerLevel",
    "AchievementsUnlocked"
]


# Define the nominal categorical features.
# These categories do NOT have a natural order.
nominal_features = [
    "Gender",
    "Location",
    "GameGenre"
]


# Define the ordinal categorical feature.
# Game difficulty has a natural order:
# Easy < Medium < Hard.
ordinal_features = [
    "GameDifficulty"
]


# Create an ordinal encoder for GameDifficulty.
# The categories are explicitly ordered so the model understands
# that Easy is lower than Medium, and Medium is lower than Hard.
difficulty_encoder = OrdinalEncoder(
    categories=[["Easy", "Medium", "Hard"]]
)


# Create the preprocessing transformer.
# Each group of columns receives the appropriate transformation.
preprocessor = ColumnTransformer(
    transformers=[

        # Standardize numerical features.
        # StandardScaler changes them to a comparable scale.
        ("num", StandardScaler(), numeric_features),

        # Convert nominal categories into binary columns.
        # handle_unknown="ignore" prevents errors if a new category
        # appears when the Streamlit app makes a prediction later.
        ("nominal", OneHotEncoder(handle_unknown="ignore"), nominal_features),

        # Convert GameDifficulty into ordered numerical values.
        ("ordinal", difficulty_encoder, ordinal_features)
    ]
)


# Convert the target labels into ordered numerical values.
# Low = 0, Medium = 1, High = 2.
# This preserves the natural order of the engagement levels.
target_mapping = {
    "Low": 0,
    "Medium": 1,
    "High": 2
}

# Apply the mapping to the target variable.
y = y.map(target_mapping)


# Display the first few converted target values to verify the encoding.
print("Encoded target values:")
print(y.head())

In [ ]:
# Import train_test_split from Scikit-learn.
# It is used to divide our dataset into training and testing sets.
from sklearn.model_selection import train_test_split


# Split the features and target into training and testing data.
X_train, X_test, y_train, y_test = train_test_split(
    X,                  # Input features.
    y,                  # Target variable.
    test_size=0.20,     # Keep 20% of the data for final testing.
    random_state=42,    # Makes the split reproducible.
    stratify=y          # Keeps Low/Medium/High proportions similar in both sets.
)


# Display the sizes of the resulting datasets.
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

In [ ]:
# Calculate the percentage of each class in the training data.
print("Training class distribution:")
print(y_train.value_counts(normalize=True) * 100)


# Calculate the percentage of each class in the testing data.
print("\nTesting class distribution:")
print(y_test.value_counts(normalize=True) * 100)

In [ ]:
# Import the four classification algorithms we want to compare.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Import metrics that will help us compare the models.
from sklearn.metrics import accuracy_score, f1_score


# Create a dictionary containing our four baseline models.
# Keeping them in a dictionary allows us to train and compare
# every model using the same code.
models = {

    # Logistic Regression provides a simple baseline.
    "Logistic Regression": LogisticRegression(
        max_iter=1000,       # Allow enough iterations for convergence.
        random_state=42      # Make results reproducible.
    ),

    # Random Forest combines many decision trees.
    "Random Forest": RandomForestClassifier(
        n_estimators=200,    # Build 200 decision trees.
        random_state=42,     # Make results reproducible.
        n_jobs=-1            # Use all available CPU cores.
    ),

    # Gradient Boosting builds trees sequentially.
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42      # Make results reproducible.
    ),

    # XGBoost is an optimized gradient-boosting algorithm.
    "XGBoost": XGBClassifier(
        n_estimators=200,    # Number of boosting trees.
        max_depth=6,         # Maximum depth of each tree.
        learning_rate=0.1,   # Controls how strongly each tree contributes.
        random_state=42,     # Make results reproducible.
        eval_metric="mlogloss",  # Evaluation metric for multiclass training.
        n_jobs=-1            # Use all available CPU cores.
    )
}


# Create an empty list to store the results of each model.
baseline_results = []


# Train and evaluate each model one at a time.
for model_name, model in models.items():

    # Create a pipeline containing our preprocessing and the current model.
    # The preprocessing is fitted ONLY using X_train.
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    # Train the complete pipeline using the training data.
    pipeline.fit(X_train, y_train)

    # Generate predictions for the unseen test data.
    y_pred = pipeline.predict(X_test)

    # Calculate accuracy.
    accuracy = accuracy_score(y_test, y_pred)

    # Calculate macro F1.
    # "macro" gives equal importance to Low, Medium, and High classes.
    f1_macro = f1_score(y_test, y_pred, average="macro")

    # Store the results for this model.
    baseline_results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "F1-Macro": f1_macro
    })


# Convert the results into a DataFrame for easy comparison.
baseline_results_df = pd.DataFrame(baseline_results)

# Sort models by accuracy from highest to lowest.
baseline_results_df = baseline_results_df.sort_values(
    by="Accuracy",
    ascending=False
)

# Reset the row numbers after sorting.
baseline_results_df = baseline_results_df.reset_index(drop=True)

# Display the baseline comparison.
baseline_results_df

In [ ]:
# Import GridSearchCV for systematically testing hyperparameter combinations.
from sklearn.model_selection import GridSearchCV

# Create the XGBoost model that GridSearchCV will tune.
xgb_model = XGBClassifier(
    random_state=42,          # Makes the results reproducible.
    eval_metric="mlogloss",   # Suitable evaluation metric for multiclass classification.
    n_jobs=-1                 # Use all available CPU cores.
)

# Create a pipeline so preprocessing happens correctly inside each
# cross-validation fold and prevents data leakage.
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),  # Apply our preprocessing steps.
    ("model", xgb_model)             # Train XGBoost after preprocessing.
])

# Define the hyperparameter values we want to test.
# GridSearchCV will test every combination of these values.
param_grid = {
    "model__n_estimators": [100, 200, 300],      # Number of boosting trees.
    "model__max_depth": [3, 5, 7],               # Maximum tree depth.
    "model__learning_rate": [0.05, 0.1, 0.2],   # Learning step size.
    "model__subsample": [0.8, 1.0],             # Fraction of rows used per boosting round.
    "model__colsample_bytree": [0.8, 1.0]       # Fraction of features used per tree.
}

# Create the GridSearchCV object.
grid_search = GridSearchCV(
    estimator=xgb_pipeline,       # The complete preprocessing + XGBoost pipeline.
    param_grid=param_grid,        # Hyperparameter combinations to test.
    scoring="f1_macro",           # Select the combination with the best macro F1.
    cv=5,                         # Use 5-fold cross-validation.
    n_jobs=-1,                    # Use all available CPU cores.
    verbose=1                     # Show progress while the search is running.
)

# Start the hyperparameter search using ONLY the training data.
grid_search.fit(X_train, y_train)

# Display the best hyperparameter combination discovered by GridSearchCV.
print("Best Parameters:")
print(grid_search.best_params_)

# Display the best cross-validation macro F1 score.
print("\nBest Cross-Validation F1-Macro:")
print(grid_search.best_score_)

In [ ]:
# Import the evaluation metrics we need.
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    ConfusionMatrixDisplay
)

# Get the best pipeline discovered by GridSearchCV.
# This includes both preprocessing and the tuned XGBoost model.
best_model = grid_search.best_estimator_

# Generate predictions for the completely unseen test dataset.
y_pred = best_model.predict(X_test)

# Generate prediction probabilities for ROC-AUC.
# Each row contains the probability of Low, Medium, and High.
y_proba = best_model.predict_proba(X_test)

# Calculate overall accuracy.
accuracy = accuracy_score(y_test, y_pred)

# Calculate macro F1.
# Each class receives equal importance regardless of its size.
f1_macro = f1_score(y_test, y_pred, average="macro")

# Calculate multiclass ROC-AUC using the One-vs-Rest approach.
roc_auc = roc_auc_score(
    y_test,
    y_proba,
    multi_class="ovr",
    average="macro"
)

# Print the main evaluation results.
print("Final Model Evaluation")
print("----------------------")
print(f"Accuracy:  {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"F1-Macro:  {f1_macro:.4f} ({f1_macro * 100:.2f}%)")
print(f"ROC-AUC:   {roc_auc:.4f}")

In [ ]:
# Generate a detailed classification report.
# This shows precision, recall, and F1-score separately
# for Low, Medium, and High engagement.
print(classification_report(
    y_test,
    y_pred,
    target_names=["Low", "Medium", "High"]
))

In [ ]:
# Calculate the confusion matrix.
# Rows represent the actual classes and columns represent predictions.
cm = confusion_matrix(y_test, y_pred)

# Create a confusion-matrix visualization.
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Low", "Medium", "High"]
)

# Plot the confusion matrix.
disp.plot()

# Add a title.
plt.title("XGBoost Confusion Matrix")

# Display the chart.
plt.show()

In [ ]:
# Get the fitted preprocessing component from our best pipeline.
fitted_preprocessor = best_model.named_steps["preprocessor"]

# Get the trained XGBoost model from our best pipeline.
trained_xgb = best_model.named_steps["model"]

# Get the names of the numerical features after preprocessing.
numeric_names = numeric_features

# Get the names created by OneHotEncoder for the categorical features.
# For example, Gender may become Gender_Female and Gender_Male.
nominal_names = fitted_preprocessor.named_transformers_[
    "nominal"
].get_feature_names_out(nominal_features)

# Keep the original name for the ordinal GameDifficulty feature.
ordinal_names = ordinal_features

# Combine all transformed feature names into one list.
feature_names = list(numeric_names) + list(nominal_names) + list(ordinal_names)

# Get the importance value assigned to every feature by XGBoost.
importances = trained_xgb.feature_importances_

# Create a DataFrame so we can easily compare features and importance.
feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

# Sort features from most important to least important.
feature_importance_df = feature_importance_df.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

# Display the feature importance table.
feature_importance_df

In [ ]:
# Select the 15 most important features.
top_features = feature_importance_df.head(15)

# Create a horizontal bar chart.
plt.figure(figsize=(10, 7))

# Plot feature importance from highest to lowest.
sns.barplot(
    data=top_features,
    x="Importance",
    y="Feature"
)

# Add a descriptive title.
plt.title("Top 15 Features Driving Player Engagement")

# Label the x-axis.
plt.xlabel("XGBoost Feature Importance")

# Label the y-axis.
plt.ylabel("Feature")

# Display the chart.
plt.show()

In [ ]:
# Import joblib, which is used to save and load Python ML objects.
import joblib

# Import os so we can create the models folder if it does not already exist.
import os

# Create the "models" folder if it doesn't exist.
# exist_ok=True prevents an error if the folder already exists.
os.makedirs("models", exist_ok=True)

# Define the location where we will save the complete trained pipeline.
model_path = "models/player_engagement_xgboost.pkl"

# Save the best tuned pipeline to the specified file.
# This includes preprocessing + encoding + scaling + XGBoost model.
joblib.dump(best_model, model_path)

# Confirm that the model was saved successfully.
print("Model saved successfully!")

# Display the exact location of the saved model.
print("Saved at:", model_path)

In [ ]:
# Load the saved ML pipeline from the file.
loaded_model = joblib.load(model_path)

# Confirm that the model was loaded successfully.
print("Saved model loaded successfully!")

# Test the loaded model on the test data.
# This confirms that the saved pipeline can still make predictions.
loaded_predictions = loaded_model.predict(X_test)

# Display the first 10 predictions.
print("Sample predictions:", loaded_predictions[:10])